# CountZES — Run Pipeline on Google Colab (Google Drive Persistent Storage)

Notebook này tự động mount Google Drive vào một thư mục project cố định:
`/content/drive/MyDrive/CountZES/`
```
MyDrive/CountZES/
├── data/      # Chứa toàn bộ datasets (FSC-147, CARPK, vgg, mbm, persenseD...)
├── weights/   # Chứa toàn bộ model weights (.pth)
└── outputs/   # Lưu ảnh trực quan hóa kết quả
```

**Trước khi chạy:**
1. Trên Colab Web: chọn **Runtime → Change runtime type → T4 GPU**.
2. Trong Antigravity IDE (với extension `Google.colab`): bấm **Select Kernel → Colab → Auto Connect**.

In [ ]:
# [CELL-1] Kiểm tra GPU & thiết lập biến môi trường PyTorch
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

!nvidia-smi
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# [CELL-2] Clone hoặc cập nhật code mới nhất từ GitHub của bạn (hỗ trợ cả repo Public & Private)
import getpass, subprocess

REPO_OWNER_PATH = "github.com/duylw/cv-advanced.git"
REPO_URL = f"https://{REPO_OWNER_PATH}"
REPO_DIR = "/content/cv-advanced"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    env_no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], env=env_no_prompt)
    if res.returncode != 0:
        print("Repo đang ở chế độ Private -> nhập GitHub PAT (token không được lưu vào file):")
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
        subprocess.run(["git", "clone", f"https://{token}@{REPO_OWNER_PATH}", REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    env_no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], env=env_no_prompt)
    if res.returncode != 0:
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
        subprocess.run(["git", "-C", REPO_DIR, "pull", f"https://{token}@{REPO_OWNER_PATH}", "main"], check=True)

%cd {REPO_DIR}
!git log -n 2 --oneline

In [ ]:
# [CELL-3] Cài các thư viện cần thiết (tương thích Python/PyTorch sẵn có của Colab, tự bỏ qua nếu đã cài)
!python -c "import addict, yapf, pycocotools, termcolor, yaml, gdown, supervision, timm, einops, transformers; assert int(transformers.__version__.split('.')[0]) < 5" 2>/dev/null || pip install -q opencv-python tqdm matplotlib timm einops supervision sentencepiece accelerate scipy pandas scikit-image addict yapf pycocotools termcolor pyyaml gdown "transformers>=4.31.0,<5.0.0"
!python -c "import segment_anything" 2>/dev/null || pip install -q git+https://github.com/facebookresearch/segment-anything.git

!python -m py_compile countzes_core.py countzes_fsc147.py countzes_carpk.py countzes_persense_D.py countzes_mbm.py countzes_vgg.py

In [ ]:
# [CELL-4] Mount Google Drive vào thư mục project chứa cả data, weights & outputs
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CountZES"
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data")
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "weights")
DRIVE_OUTPUTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "outputs")

for d in [DRIVE_PROJECT_DIR, DRIVE_DATA_DIR, DRIVE_WEIGHTS_DIR, DRIVE_OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Copy metadata (.json/.txt) từ repo sang Drive nếu Drive chưa có (idempotent, không ghi đè)
for root, _, files_in_dir in os.walk("/content/cv-advanced/data", followlinks=False):
    rel_root = os.path.relpath(root, "/content/cv-advanced/data")
    target_root = os.path.join(DRIVE_DATA_DIR, rel_root) if rel_root != "." else DRIVE_DATA_DIR
    os.makedirs(target_root, exist_ok=True)
    for fname in files_in_dir:
        src_f = os.path.join(root, fname)
        dst_f = os.path.join(target_root, fname)
        if not os.path.islink(src_f) and not os.path.exists(dst_f):
            shutil.copy2(src_f, dst_f)

# Liên kết các thư mục dataset trong repo sang Drive để mọi đường dẫn mặc định ./data/<dataset> đều đọc từ Drive
for sub in ["vgg", "mbm", "persenseD"]:
    os.makedirs(os.path.join(DRIVE_DATA_DIR, sub), exist_ok=True)
    local_sub = os.path.join("data", sub)
    if os.path.islink(local_sub):
        os.unlink(local_sub)
    elif os.path.exists(local_sub):
        shutil.rmtree(local_sub)
    os.symlink(os.path.join(DRIVE_DATA_DIR, sub), local_sub)

for rel_sub in ["FSC-147/images_384_VarV2", "CARPK/Images", "CARPK/gt_density_map"]:
    drive_sub = os.path.join(DRIVE_DATA_DIR, rel_sub)
    os.makedirs(drive_sub, exist_ok=True)
    local_sub = os.path.join("data", rel_sub)
    os.makedirs(os.path.dirname(local_sub), exist_ok=True)
    if os.path.islink(local_sub):
        os.unlink(local_sub)
    elif os.path.exists(local_sub):
        shutil.rmtree(local_sub)
    os.symlink(drive_sub, local_sub)

# Liên kết ./outputs sang Drive để không mất ảnh kết quả
if os.path.islink("outputs"):
    os.unlink("outputs")
elif os.path.exists("outputs"):
    shutil.rmtree("outputs")
os.symlink(DRIVE_OUTPUTS_DIR, "outputs")

print("Mounted project folder:", DRIVE_PROJECT_DIR)
print(" - Data   :", DRIVE_DATA_DIR)
print(" - Weights:", DRIVE_WEIGHTS_DIR)
print(" - Outputs:", DRIVE_OUTPUTS_DIR)

In [ ]:
# [CELL-5] Tải Model Weights vào Drive (nếu chưa có) và symlink vào đúng đường dẫn repo
os.makedirs("DSALVANet/checkpoints", exist_ok=True)
os.makedirs("GroundingDINO/weights", exist_ok=True)
os.makedirs("CounTR/output_allnew_dir", exist_ok=True)

def ensure_weight(weight_name, local_path, download_cmd=None):
    drive_path = os.path.join(DRIVE_WEIGHTS_DIR, weight_name)
    if not (os.path.exists(drive_path) and os.path.getsize(drive_path) > 0):
        if download_cmd:
            print(f"[Download -> Drive] {drive_path}")
            get_ipython().system(download_cmd.format(dst=drive_path))
        else:
            print(f"[Optional] Chưa có {weight_name} trong {DRIVE_WEIGHTS_DIR}")
            return
    else:
        print(f"[Drive OK] {drive_path}")

    if os.path.islink(local_path) or os.path.exists(local_path):
        os.remove(local_path)
    os.symlink(drive_path, local_path)
    print(f"  -> Linked {local_path}")

# 1. SAM ViT-H (~2.56 GB)
ensure_weight(
    "sam_vit_h_4b8939.pth",
    "data/sam_vit_h_4b8939.pth",
    "wget -q --show-progress -O '{dst}' https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
)

# 2. DSALVANet pretrained FSC-147
ensure_weight(
    "checkpoint_200.pth",
    "DSALVANet/checkpoints/checkpoint_200.pth",
    "gdown 1julzH9MJSK1xTGchb1r0CXdZ2wzF5-Kp -O '{dst}'"
)

# 3. GroundingDINO SwinT-OGC
ensure_weight(
    "groundingdino_swint_ogc.pth",
    "GroundingDINO/weights/groundingdino_swint_ogc.pth",
    "gdown 13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g -O '{dst}'"
)

# 4. CounTR weights (nếu bạn tự upload FSC147.pth / CARPK.pth vào MyDrive/CountZES/weights/)
ensure_weight("FSC147.pth", "CounTR/output_allnew_dir/FSC147.pth")
ensure_weight("CARPK.pth", "CounTR/output_allnew_dir/CARPK.pth")

In [ ]:
# [CELL-6] Tải dataset mẫu vào Google Drive (nếu chưa có sẵn trong MyDrive/CountZES/data)
import glob
DATASET_TO_FETCH = "vgg"  # "vgg" | "mbm" | "none" (chọn "none" nếu bạn đã tự chép dataset lên Drive)

def fetch_and_normalize_dataset(ds_name, file_id):
    target_images_dir = os.path.join(DRIVE_DATA_DIR, ds_name, "Images")
    if os.path.exists(target_images_dir) and any(os.scandir(target_images_dir)):
        print(f"[Drive OK] Đã có sẵn dataset: {target_images_dir}")
        return
    zip_path = f"/content/{ds_name}.zip"
    tmp_dir = f"/content/_tmp_{ds_name}"
    shutil.rmtree(tmp_dir, ignore_errors=True)
    get_ipython().system(f"gdown {file_id} -O {zip_path}")
    get_ipython().system(f"unzip -q -o {zip_path} -d {tmp_dir}")

    # Tự động tìm thư mục chứa các cặp ảnh + mask (*A.png) và chuẩn hóa về data/<ds>/Images/<class>/
    mask_files = glob.glob(os.path.join(tmp_dir, "**", "*A.png"), recursive=True)
    assert mask_files, f"Không tìm thấy file *A.png sau khi giải nén {ds_name}.zip"
    class_dirs = sorted({os.path.dirname(p) for p in mask_files})
    os.makedirs(target_images_dir, exist_ok=True)
    for cdir in class_dirs:
        cname = os.path.basename(cdir)
        if cname.lower() in {"images", ds_name.lower()}:
            cname = "cells" if ds_name == "vgg" else "blood cells"
        dst_cdir = os.path.join(target_images_dir, cname)
        os.makedirs(dst_cdir, exist_ok=True)
        for f in os.listdir(cdir):
            src_f = os.path.join(cdir, f)
            if os.path.isfile(src_f):
                shutil.copy2(src_f, os.path.join(dst_cdir, f))
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"[Dataset Ready] {target_images_dir}")

if DATASET_TO_FETCH == "vgg":
    fetch_and_normalize_dataset("vgg", "1FerZjhpA1MCtsmjD-eF1byZvOWEDnwZL")
elif DATASET_TO_FETCH == "mbm":
    fetch_and_normalize_dataset("mbm", "1i54Hcw2GbCVp0Kz7jei2Lr7Cmi2DZkol")

!find "{DRIVE_DATA_DIR}" -maxdepth 4

In [ ]:
# [CELL-7] Chạy pipeline đánh giá chính thức (đọc data & lưu outputs thẳng trên Drive)
SCRIPT = "countzes_vgg.py"       # countzes_vgg.py | countzes_mbm.py | countzes_fsc147.py | countzes_carpk.py | countzes_persense_D.py
FSOC = "DSALVANet"               # "DSALVANet" hoặc "countr"
OUTDIR = "colab_run_default"
EXTRA_FLAGS = "--visualize"      # Thêm "--fix_sses_inversion --single_instance_mode unit" để A/B test P2

!python {SCRIPT} --data "{DRIVE_DATA_DIR}" --fsoc {FSOC} --outdir {OUTDIR} {EXTRA_FLAGS}

In [ ]:
# [CELL-8] Xem nhanh ảnh kết quả đã lưu trên Drive (outputs/)
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob(f"{DRIVE_OUTPUTS_DIR}/{OUTDIR}/**/*.png", recursive=True))[:6]
print(f"Tìm thấy {len(imgs)} ảnh hiển thị mẫu trong {DRIVE_OUTPUTS_DIR}/{OUTDIR}:")
for p in imgs:
    print(p)
    display(Image(filename=p))